In [1]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
WESAD_PATH = r"C:\Users\gloriosog\OneDrive - Milwaukee School of Engineering\Year 4 Courses\Semester 1\Senior Design\WESAD Dataset\WESAD2\WESAD"
SUBJECTS = ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']
FS_ACC = 32
FS_LABEL = 700
LABEL_NAMES = {1: 'Baseline', 2: 'Stress', 3: 'Amusement', 4: 'Meditation'}

def align_labels(signal_length, signal_fs, labels, label_fs):
    signal_times = np.arange(signal_length) / signal_fs
    label_times = np.arange(len(labels)) / label_fs
    indices = np.searchsorted(label_times, signal_times)
    return labels[np.clip(indices, 0, len(labels) - 1)]

In [3]:
records = []

for subject in SUBJECTS:
    file_path = f"{WESAD_PATH}\\{subject}\\{subject}.pkl"
    print(f"Loading {subject}...")
    with open(file_path, 'rb') as f:
        data = pickle.load(f, encoding='latin1')

    acc = data['signal']['wrist']['ACC']           # shape (N, 3)
    labels_raw = data['label'].flatten()

    # Compute magnitude
    magnitude = np.sqrt((acc ** 2).sum(axis=1))

    # Align 700Hz labels → 32Hz ACC
    aligned = align_labels(len(magnitude), FS_ACC, labels_raw, FS_LABEL)

    for mag, lbl in zip(magnitude, aligned):
        if lbl in LABEL_NAMES:
            records.append({'subject': subject, 'condition': LABEL_NAMES[lbl], 'magnitude': mag})

df = pd.DataFrame(records)
print(f"\nLoaded {len(df):,} samples across {df['subject'].nunique()} subjects")
print(df.groupby('condition')['magnitude'].count())


Loading S2...


C:\Users\gloriosog\AppData\Local\Temp\ipykernel_32776\1523925767.py:7: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  data = pickle.load(f, encoding='latin1')


Loading S3...
Loading S4...
Loading S5...
Loading S6...
Loading S7...
Loading S8...
Loading S9...
Loading S10...
Loading S11...
Loading S13...
Loading S14...
Loading S15...
Loading S16...
Loading S17...

Loaded 1,438,656 samples across 15 subjects
condition
Amusement     178400
Baseline      563552
Meditation    377792
Stress        318912
Name: magnitude, dtype: int64


In [9]:
# Binary remap: Stress vs Non-Stress
df['label'] = df['condition'].apply(lambda x: 'Stress' if x == 'Stress' else 'Non-Stress')

summary_records = []
for subject in SUBJECTS:
    subj_df = df[df['subject'] == subject]
    for condition in ['Stress', 'Non-Stress']:
        vals = subj_df[subj_df['label'] == condition]['magnitude'].values
        summary_records.append({
            'Subject':    subject,
            'Condition':  condition,
            'N Samples':  len(vals),
            'Mean (g)':   round(vals.mean(), 4),
            'Median (g)': round(np.median(vals), 4),
            'Std (g)':    round(vals.std(), 4),
            'P95 (g)':    round(np.percentile(vals, 95), 4),
            'Max (g)':    round(vals.max(), 4),
        })

summary = pd.DataFrame(summary_records)
print(summary.to_string(index=False))


Subject  Condition  N Samples  Mean (g)  Median (g)  Std (g)  P95 (g)  Max (g)
     S2     Stress      19680   62.9370     62.7136   2.7372  66.0534 145.9932
     S2 Non-Stress      72768   63.3397     63.1823   2.4882  64.8228 185.6987
     S3     Stress      20480   65.0649     64.6375   6.3871  72.1388 167.0299
     S3 Non-Stress      73440   64.6121     64.7302   1.7305  65.7875 115.3473
     S4     Stress      20320   64.8583     64.5407   4.1717  68.1616 201.7969
     S4 Non-Stress      74720   64.7450     64.7688   1.5284  65.9469 131.6245
     S5     Stress      20640   62.5844     62.5939   1.0596  63.7652 102.4353
     S5 Non-Stress      75712   63.1592     63.1348   1.6629  63.9140 109.5536
     S6     Stress      20800   62.9956     62.9365   2.3681  65.2763 125.0520
     S6 Non-Stress      74848   63.0162     62.9762   1.7377  64.3351 111.8347
     S7     Stress      20480   63.3167     62.8967   7.1682  70.5195 146.5264
     S7 Non-Stress      75136   62.8747     62.8808 

In [ ]:
# Load S4 for time-series visualization
with open(f"{WESAD_PATH}\\S4\\S4.pkl", 'rb') as f:
    s4 = pickle.load(f, encoding='latin1')

acc_s4 = s4['signal']['wrist']['ACC']
labels_s4 = s4['label'].flatten()
magnitude_s4 = np.sqrt((acc_s4 ** 2).sum(axis=1))
aligned_s4 = align_labels(len(magnitude_s4), FS_ACC, labels_s4, FS_LABEL)
time_s4 = np.arange(len(magnitude_s4)) / FS_ACC

plt.figure(figsize=(16, 4))
plt.plot(time_s4, magnitude_s4, color='black', linewidth=0.5, alpha=0.8)

diff = np.diff(aligned_s4, prepend=-1)
change_idx = np.append(np.where(diff != 0)[0], len(aligned_s4))
added = set()
for i in range(len(change_idx) - 1):
    lbl = aligned_s4[change_idx[i]]
    if lbl in LABEL_NAMES:
        t0 = change_idx[i] / FS_ACC
        t1 = change_idx[i+1] / FS_ACC
        lab_str = LABEL_NAMES[lbl] if lbl not in added else None
        plt.axvspan(t0, t1, color=colors[LABEL_NAMES[lbl]], alpha=0.2, label=lab_str)
        added.add(lbl)

plt.title('S4 — ACC Magnitude Over Time with Condition Labels')
plt.xlabel('Time (seconds)')
plt.ylabel('Magnitude (g)')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


C:\Users\gloriosog\AppData\Local\Temp\ipykernel_32776\3524328354.py:3: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  s4 = pickle.load(f, encoding='latin1')


In [8]:
# Binary labels: 2 = Stress, everything else non-zero = Non-Stress (skip 0/transient)
valid_mask = aligned != 0
magnitude = magnitude[valid_mask]
aligned = aligned[valid_mask]

stress_mask = aligned == 2
nonstress_mask = ~stress_mask

for condition, mask in [('Stress', stress_mask), ('Non-Stress', nonstress_mask)]:
    vals = magnitude[mask]
    all_records.append({
        'Subject':    subject,
        'Condition':  condition,
        'N Samples':  len(vals),
        'Mean (g)':   round(vals.mean(), 4),
        'Median (g)': round(np.median(vals), 4),
        'Std (g)':    round(vals.std(), 4),
        'P95 (g)':    round(np.percentile(vals, 95), 4),
        'P99 (g)':    round(np.percentile(vals, 99), 4),
        'Max (g)':    round(vals.max(), 4),
    })

summary = pd.DataFrame(all_records)
pd.set_option('display.max_rows', 100)
print(summary.to_string(index=False))

NameError: name 'all_records' is not defined